In [ ]:
import os
import pandas as pd
from datetime import datetime 
import duckdb
import unicodedata
import sys
from pathlib import Path
from kedro.framework.startup import bootstrap_project
from kedro.framework.session import KedroSession

# 1. Move to project root if we are in the notebooks folder
if Path.cwd().name == "notebooks":
    os.chdir("..")

# 2. Initialize Kedro
project_path = Path.cwd()
bootstrap_project(project_path)

# 3. Create session and get catalog
session = KedroSession.create(project_path)
context = session.load_context()
catalog = context.catalog

print(f"✅ Kedro context loaded! Project root: {project_path}")

[04/21/26 22:57:33] INFO     Using                                                                  ]8;id=855141;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\framework\project\__init__.py\__init__.py]8;;\:]8;id=302056;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\framework\project\__init__.py#269\269]8;;\
                             'c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\framework\                
                             project\rich_logging.yml' as logging configuration.                                   

[04/21/26 22:57:33] WARNING  c:\Users\User\miniconda3\envs\unis\Lib\site-packages\requests\__init__ ]8;id=655313;file://c:\Users\User\miniconda3\envs\unis\Lib\warnings.py\warnings.py]8;;\:]8;id=233580;file://c:\Users\User\miniconda3\envs\unis\Lib\warnings.py#110\110]8;;\
                             .py:113: RequestsDependencyWarning: urllib3 (2.6.1) or chardet                        
                             (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported                    
                             version!                                                                              
                               warnings.warn(                                                                      
                                                                                                                   

[04/21/26 22:57:34] INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=855682;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro_telemetry\plugin.py\plugin.py]8;;\:]8;id=748034;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro_telemetry\plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

✅ Kedro context loaded! Project root: g:\Unidades compartidas\Alianzas\3. Data\UNIS\unis-perm-flow


In [ ]:
# Add the 'src' directory to the path
sys.path.append(os.path.abspath("src"))
import unis_perm_flow.pipelines.data_processing.nodes as nodes_dproc
import unis_perm_flow.pipelines.calac_activos_baj_grad.nodes as nodes_abg

# Importación de fuentes

In [ ]:
unis_calaca_ext  = catalog.load('unis_calendario_extendido')
unis_calaca_uptoday  = catalog.load('unis_calendario_extendido_uptoday')
unis_estaca_sd = catalog.load('unis_estaca_sd')

[04/21/26 22:57:35] INFO     Loading data from unis_calendario_extendido (ParquetDataset)...   ]8;id=199053;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=526649;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

                    INFO     Loading data from unis_calendario_extendido_uptoday               ]8;id=896838;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=841977;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

                    INFO     Loading data from unis_estaca_sd (ParquetDataset)...              ]8;id=35592;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=436956;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

# Importación de parámetros

In [ ]:
parameters = catalog.load("parameters")

[04/21/26 22:57:50] INFO     Loading data from parameters (MemoryDataset)...                   ]8;id=308552;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=858808;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

In [ ]:
# bajas_calac
bajas_calac = parameters['bajas_calac']
graduados_calac = parameters['graduados_calac']

# Bajas

In [ ]:
bajas_calendario_academico = nodes_abg.momento_baja(
    # 1. El DataFrame (Objeto en memoria)
    unis_estaca_sd=unis_estaca_sd, 
    dict_duracion=parameters['graduados_calac']['dict_niveles_duracion'],
    fallback_weeks=parameters['graduados_calac']['graduation_fallback_weeks'],
    
    # 2. Valores extraídos de diccionarios de configuración
    unis_col_fechadef=bajas_calac['unis_col_fechadef'],
    unis_col_fechatemp=parameters['bajas_calac']['unis_col_fechatemp'],
    
    # 3. Nombre del dataset según el catálogo
    unis_calaca= unis_calaca_uptoday.drop(columns= 'cohorte'),
    
    # 4. Parámetros de transformación y cruce
    left_on=parameters['bajas_calac']['merge_left_on'],
    right_on=parameters['bajas_calac']['merge_right_on'],
    group_key=parameters['bajas_calac']['unis_calaca_col_cohorte_inicial'],
    sort_cols=parameters['bajas_calac']['unis_calaca_col_sort']
)

In [ ]:
# 1. Obtenemos el Top 10 (mismo proceso que antes)
top_diez_desercion = (
    bajas_calendario_academico
    .loc[:, ['periodo_inicial','nivel', 'fecha_ingreso', 'semana_acumulada', 'di']]
    .groupby(['periodo_inicial','nivel',  'fecha_ingreso', 'semana_acumulada'])
    .agg({'di': 'sum'})
    .reset_index()
    .sort_values(by='di', ascending=False)
    .head(10)
)

# 2. Aplicamos las barras de datos de color rojo
top_diez_estilado = top_diez_desercion.style.bar(
    subset=['di'], 
    color='#f8766d', # Un rojo suave tipo "soft red"
    align='left',    # Las barras nacen desde la izquierda
    vmin=0           # El mínimo es cero para que la escala sea real
)

# Visualizar en el Notebook
top_diez_estilado

,periodo_inicial,nivel,fecha_ingreso,semana_acumulada,di
49,9251.000000,maestria,2025-01-14 00:00:00,23.000000,61
73,9252.000000,maestria,2025-05-13 00:00:00,27.000000,33
86,9253.000000,maestria,2025-09-02 00:00:00,10.000000,26
31,9243.000000,maestria,2024-08-27 00:00:00,63.000000,26
112,9261.000000,maestria,2026-01-13 00:00:00,9.000000,18
34,9251.000000,maestria,2025-01-14 00:00:00,3.000000,17
72,9252.000000,maestria,2025-05-13 00:00:00,26.000000,16
90,9253.000000,maestria,2025-09-02 00:00:00,14.000000,16
26,9243.000000,maestria,2024-08-27 00:00:00,40.000000,15
102,9254.000000,maestria,2025-10-28 00:00:00,14.000000,11


# Graduados

In [ ]:
graduados_calendario_academico = nodes_abg.momento_grado(
    # 1. Referencias a datasets/DataFrames
    unis_estaca=unis_estaca_sd,
    unis_calaca=unis_calaca_uptoday.drop(columns= 'cohorte'),
    
    # 2. Configuración de lógica de negocio (Duraciones y Grado Inmediato)
   
    col_gi=parameters['graduados_calac']['graduation_col_gi'],
    
    # 3. Parámetro de seguridad (Semanas por defecto si falla el cálculo)
    dict_duracion=parameters['graduados_calac']['dict_niveles_duracion'],
    fallback_weeks=parameters['graduados_calac']['graduation_fallback_weeks'],
    
    # 4. Llaves para el cruce de tablas (Joins)
    join_left=parameters['graduados_calac']['graduation_join_keys_left'],
    join_right=parameters['graduados_calac']['graduation_join_keys_right']
)

In [ ]:
# 1. Obtenemos el Top 10 (mismo proceso que antes)
top_diez_graduacion = (
    graduados_calendario_academico
    .loc[:, ['periodo_inicial','nivel', 'fecha_ingreso', 'semana_acumulada', 'gi']]
    .groupby(['periodo_inicial','nivel',  'fecha_ingreso', 'semana_acumulada'])
    .agg({'gi': 'sum'})
    .reset_index()
    .sort_values(by='gi', ascending=False)
    .head(100)
)

# 2. Aplicamos las barras de datos de color rojo
top_diez_estilado = top_diez_graduacion.style.bar(
    subset=['gi'], 
    color="#41747A", # Un verde suave tipo "soft green"
    align='left',    # Las barras nacen desde la izquierda
    vmin=0           # El mínimo es cero para que la escala sea real
)

# Visualizar en el Notebook
top_diez_estilado

,periodo_inicial,nivel,fecha_ingreso,semana_acumulada,gi


# Activos

In [ ]:
activos_calendario_academico = nodes_abg.momento_activos(
   unis_estaca=unis_estaca_sd,
    unis_calaca=unis_calaca_uptoday.drop(columns= 'cohorte'),
    
    # 2. Configuración de lógica de negocio (Duraciones y Grado Inmediato)
    dict_duracion=parameters['graduados_calac']['dict_niveles_duracion'],
    col_di='di',
    col_gi='gi',
    
    # 3. Parámetro de seguridad (Semanas por defecto si falla el cálculo)
    fallback_weeks=None,
    
    # 4. Llaves para el cruce de tablas (Joins)
    join_left='fecha_activo',
    join_right='fecha_inicio',
    group_key = 'fecha_ingreso'
) 

In [ ]:
activos_calendario_academico.columns


Index(['id_inconcert', 'identificacion', 'correo_2', 'nombres',
       'fecha_registro', 'cohorte', 'fecha_ingreso', 'usuario_institucional',
       'nivel', 'programa', 'estado', 'tipo_baja', 'motivo_baja',
       'submotivo_baja', 'comentarios', 'fecha_de_baja_t', 'fecha_de_baja_d',
       'fecha_de_reingreso', 'fecha_de_grado', 'exito_estudiantil', 'alianza',
       'etapa_studen_journey', 'di', 'gi', 'fecha_activo',
       'max_semana_teorica', 'periodo_raw', 'periodo_inicial',
       'periodo_actual', 'sesion', 'fecha_inicio', 'fecha_fin',
       'shifted_fecha_inicio', 'semana', 'semana_acumulada', 'month',
       'mes_academico', 'anio_gregoriano', 'mes_gregoriano', 'student_journey',
       'tipo', 'engi', 'ai'],
      dtype='str')

In [ ]:
unis_estados_calac = catalog.load('unis_estados_calac')